In [2]:
import os
import random
import re
import pandas as pd
from PIL import Image
from vllm import LLM, SamplingParams
import warnings
warnings.filterwarnings('ignore')


MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"
TP_SIZE = 2  
IMAGE_BASE_DIR = "relation_image_exports" 
FEW_SHOT_COUNT = 10

In [3]:
print(f"Loading VLM {MODEL_PATH} ... ")
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TP_SIZE,
    dtype="bfloat16",
    max_model_len=4096,           # 视觉模型需要较大上下文
    gpu_memory_utilization=0.85,
    enforce_eager=True,
    limit_mm_per_prompt={"image": FEW_SHOT_COUNT} # 限制每条Prompt的图片数量
)
print(f"Loaded VLM {MODEL_PATH} successfully")

Loading VLM Qwen/Qwen2.5-VL-7B-Instruct ... 
INFO 05-21 09:59:31 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 10}, 'model': 'Qwen/Qwen2.5-VL-7B-Instruct'}
INFO 05-21 09:59:33 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 05-21 09:59:33 [model.py:1678] Using max model len 4096
INFO 05-21 09:59:33 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-21 09:59:33 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-21 09:59:33 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-21 09:59:33 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


(EngineCore pid=582443) INFO 05-21 09:59:43 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, coll

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(Worker_TP0 pid=582461) INFO 05-21 09:59:48 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/3)
(Worker_TP0 pid=582461) INFO 05-21 09:59:49 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/3)
(Worker_TP0 pid=582461) INFO 05-21 09:59:49 [weight_utils.py:825] Prefetching checkpoint files: 30% (3/3)
(Worker_TP0 pid=582461) INFO 05-21 09:59:49 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.58s
(Worker_TP0 pid=582461) INFO 05-21 09:59:50 [default_loader.py:384] Loading weights took 1.71 seconds
(Worker_TP0 pid=582461) INFO 05-21 09:59:51 [gpu_model_runner.py:4820] Model loading took 7.87 GiB memory and 3.720584 seconds
(Worker_TP0 pid=582461) INFO 05-21 09:59:51 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 video items of the maximum feature size.
(Worker_TP0 pid=582461) INFO 05-21 09:59:55 [gpu_worker.py:436] Available KV cache memory: 28.38 GiB
(EngineCore pid=582443) IN

In [65]:
test_cases = [
    {"Sub": "car", "Rel": "on", "Obj": "street", "Expected": "Yes"},

    {"Sub": "person", "Rel": "wearing", "Obj": "hat", "Expected": "Yes"},
    {"Sub": "boat", "Rel": "flying in", "Obj": "room", "Expected": "No"},
    {"Sub": "cat", "Rel": "sitting on", "Obj": "table", "Expected": "Yes"},
    {"Sub": "pizza", "Rel": "eating", "Obj": "man", "Expected": "No"},
    {"Sub": "train", "Rel": "parked on", "Obj": "track", "Expected": "Yes"},
    {"Sub": "tree", "Rel": "riding", "Obj": "bike", "Expected": "No"},
    {"Sub": "dog", "Rel": "holding", "Obj": "branch", "Expected": "Yes"},
    {"Sub": "flower", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},

    {"Sub": "logo", "Rel": "of", "Obj": "banana", "Expected": "Yes"},

    {"Sub": "person", "Rel": "along", "Obj": "towel", "Expected": "No"},
    {"Sub": "vase", "Rel": "sitting on", "Obj": "airplane", "Expected": "No"},
    {"Sub": "bag", "Rel": "across", "Obj": "street", "Expected": "No"},
    {"Sub": "leg", "Rel": "of", "Obj": "branch", "Expected": "No"},
    {"Sub": "laptop", "Rel": "of", "Obj": "screen", "Expected": "No"},
    {"Sub": "girl", "Rel": "attached to", "Obj": "book", "Expected": "No"},
    {"Sub": "man", "Rel": "eating", "Obj": "surfboard", "Expected": "No"},
    {"Sub": "man", "Rel": "says", "Obj": "snowboard", "Expected": "No"},


    {"Sub": "animal", "Rel": "above", "Obj": "ear", "Expected": "No"},           # 讨论过的空间几何错误
    {"Sub": "horse", "Rel": "riding", "Obj": "boy", "Expected": "No"},           # 主客体颠倒：人骑马，马不骑人
    {"Sub": "food", "Rel": "eating", "Obj": "woman", "Expected": "No"},          # 主客体颠倒：食物不能吃人
    {"Sub": "shoe", "Rel": "walking on", "Obj": "sidewalk", "Expected": "No"},   # 拟人化陷阱：鞋子本身不会走
    {"Sub": "elephant", "Rel": "sitting on", "Obj": "chair", "Expected": "No"},  # 物理尺度陷阱：普通椅子承受不了大象
    {"Sub": "cup", "Rel": "holding", "Obj": "elephant", "Expected": "No"},       # 物理尺度陷阱：杯子装不下大象
    {"Sub": "motorcycle", "Rel": "wearing", "Obj": "helmet", "Expected": "No"},  # 拟人化陷阱：机器不穿戴衣物
    {"Sub": "wing", "Rel": "part of", "Obj": "dog", "Expected": "No"},           # 解剖学错误：狗没有翅膀
    {"Sub": "leg", "Rel": "attached to", "Obj": "wheel", "Expected": "No"},      # 生物与机械错乱
    {"Sub": "tree", "Rel": "growing on", "Obj": "dog", "Expected": "No"},        # 生物常识错误
    {"Sub": "boat", "Rel": "parked on", "Obj": "tree", "Expected": "No"},        # 极端边缘场景
    {"Sub": "building", "Rel": "wearing", "Obj": "hat", "Expected": "No"},       # 隐喻/卡通陷阱
    {"Sub": "car", "Rel": "in", "Obj": "box", "Expected": "No"},                 # 尺度陷阱：真车装不进普通纸箱
    {"Sub": "logo", "Rel": "painted on", "Obj": "horse", "Expected": "No"},      # 讨论过的边缘特例

    # ==========================================
    # 补充 18 个 "Yes" (总计达到 25 个 Yes)
    # 包含日常常识，以及防止模型过度保守的“迷惑正确项”
    # ==========================================
    {"Sub": "bird", "Rel": "standing on", "Obj": "cow", "Expected": "Yes"},      # 自然界常见（牛背鹭），测试模型是否敢说Yes
    {"Sub": "plant", "Rel": "growing on", "Obj": "building", "Expected": "Yes"}, # 爬山虎、青苔等
    {"Sub": "sign", "Rel": "says", "Obj": "number", "Expected": "Yes"},          # 语言习惯在视觉中的体现（限速牌）
    {"Sub": "cat", "Rel": "looking at", "Obj": "book", "Expected": "Yes"},       # 猫不认字，但物理上盯着书看是完全合理的
    {"Sub": "dog", "Rel": "covered in", "Obj": "snow", "Expected": "Yes"},       # 视觉常识：狗在雪地里玩耍
    {"Sub": "tree", "Rel": "growing on", "Obj": "hill", "Expected": "Yes"},      # 标准自然场景
    {"Sub": "bag", "Rel": "hanging from", "Obj": "branch", "Expected": "Yes"},   # 包挂在树枝上，视觉合理
    {"Sub": "bird", "Rel": "flying in", "Obj": "room", "Expected": "Yes"},       # 相比船在房间飞(No)，鸟飞进房间是常见意外(Yes)
    {"Sub": "person", "Rel": "riding", "Obj": "surfboard", "Expected": "Yes"},   # 冲浪的正确描述
    {"Sub": "door", "Rel": "of", "Obj": "car", "Expected": "Yes"},               # 从属关系对照
    {"Sub": "snow", "Rel": "covering", "Obj": "car", "Expected": "Yes"},         # 典型的冬季视觉场景
    {"Sub": "boat", "Rel": "on", "Obj": "beach", "Expected": "Yes"},             # 搁浅或停靠的船
    {"Sub": "man", "Rel": "carrying", "Obj": "box", "Expected": "Yes"},          # 典型人类动作
    {"Sub": "clock", "Rel": "on", "Obj": "building", "Expected": "Yes"},         # 钟楼/建筑上的钟
    {"Sub": "umbrella", "Rel": "covering", "Obj": "woman", "Expected": "Yes"},   # 撑伞的物理状态
    {"Sub": "pizza", "Rel": "in", "Obj": "box", "Expected": "Yes"},              # 日常物品包含关系
    {"Sub": "plane", "Rel": "above", "Obj": "mountain", "Expected": "Yes"},      # 正常的宏观空间关系
    {"Sub": "chair", "Rel": "near", "Obj": "table", "Expected": "Yes"}           # 标准室内陈设
]

def format_question(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    if rel != 'and':
        if rel == 'wears': rel_formatted = 'wearing'
        elif rel == 'says': rel_formatted = 'saying'
        elif rel == 'has': rel_formatted = 'have'
        elif rel == 'riding': rel_formatted = 'riding on'
        
        rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        return f"Can {sub_f} {rel_f} {obj_f}?"
    else:
        return f"Can {sub_f} {rel} {obj_f} appear together?"

In [66]:
# 跨模态解耦：在 CoT 中绝不提“蓝框/橙框”，只谈纯物理常识
def generate_auto_cot(sub, rel, obj, is_positive_example=True):
    if is_positive_example:
        return (f"Reasoning: 1) Based on real-world knowledge, a '{sub}' and a '{obj}' are common physical entities. "
                f"2) According to physical laws and spatial possibilities, it is completely normal and realistic for a '{sub}' to be {rel} a '{obj}'. "
                f"3) This is a visually plausible scenario that easily appears in real-world photographs.\nFinal Answer: Yes")
    else:
        return (f"Reasoning: 1) Consider the physical properties, scale, and biological nature of a '{sub}' and a '{obj}'. "
                f"2) In the real world, a '{sub}' cannot logically or physically be {rel} a '{obj}'. "
                f"3) This situation violates basic commonsense and would only appear in CGI, fantasy, or metaphorical contexts.\nFinal Answer: No")

def parse_info_from_filename(filename):
    
    sub_match = re.search(r'_sub-[^_]+-([^_]+)_', filename)
    obj_match = re.search(r'_obj-[^_]+-([^_]+)_', filename)
    
    sub = sub_match.group(1).replace('_', ' ') if sub_match else "subject"
    obj = obj_match.group(1).replace('_', ' ') if obj_match else "object"
    return sub, obj

def get_folder_for_relation(rel_name, base_dir):
    # 将 "flying in" 转换为 "flying_in" 并匹配 "15_flying_in"
    safe_rel = rel_name.replace(" ", "_")
    if not os.path.exists(base_dir):
        return None
        
    for folder in os.listdir(base_dir):
        if folder.endswith(f"_{safe_rel}"):
            return os.path.join(base_dir, folder)
    return None


In [67]:

def get_dynamic_few_shots(rel_name, base_dir, num_shots=1):
    folder_path = get_folder_for_relation(rel_name, base_dir)
    if not folder_path:
        return [] # 如果没找到这个关系的文件夹，返回空列表
        
    # 只挑选带有 _boxed.jpg 或 _boxed.png 的图片
    boxed_imgs = [f for f in os.listdir(folder_path) if f.endswith('_boxed.jpg') or f.endswith('_boxed.png')]
    if not boxed_imgs:
        return []
        
    # 随机抽样
    sampled_imgs = random.sample(boxed_imgs, min(num_shots, len(boxed_imgs)))
    
    shots = []
    for img_name in sampled_imgs:
        img_path = os.path.join(folder_path, img_name)
        sub, obj = parse_info_from_filename(img_name)
        question = format_question(sub, rel_name, obj)
        
        # 因为是从 VG 数据集直接提出来的真实存在的三元组图片，必定是合乎物理法则的，所以固定是 Positive Example
        cot_text = generate_auto_cot(sub, rel_name, obj, is_positive_example=True)
        
        # 组装 VLM 格式
        shots.append({
            "role": "user",
            "content": [
                {"type": "image", "image": Image.open(img_path)},
                {"type": "text", "text": question}
            ]
        })
        shots.append({
            "role": "assistant",
            "content": cot_text
        })
    return shots

In [68]:

system_prompt_visual = """You are an expert in visual commonsense reasoning. 
You will be provided with examples consisting of an image and a question in the form: 'Can a <Subject> be <Relationship> a <Object>?'.
(Note: In the example images, bounding boxes may be present to help you identify the entities, but you must NOT rely on them for your final logical reasoning).

Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
Determine whether the situation is logically possible in the real world according to commonsense knowledge, and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations)..
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment purely on general physical laws, scales, and typical real-world conditions.
Answer step-by-step and end your response strictly with 'Final Answer: Yes' or 'Final Answer: No'."""


def run_dynamic_vlm_ablation():
    messages_batch = []
    
    for case in test_cases:
        target_rel = case['Rel']
        question = format_question(case['Sub'], target_rel, case['Obj'])
        
        # 1. 放入 System Prompt
        messages = [{"role": "system", "content": system_prompt_visual}]
        
        # 2. 动态检索并拼接 针对当前关系的 图像 Few-Shot
        dynamic_shots = get_dynamic_few_shots(target_rel, IMAGE_BASE_DIR, num_shots=FEW_SHOT_COUNT)
        if dynamic_shots:
            messages.extend(dynamic_shots)
            
        # 3. 放入真正的测试题（纯文本！这代表了真实世界的无框提问）
        messages.append({
            "role": "user", 
            "content": [{"type": "text", "text": question}]
        })
        
        messages_batch.append(messages)
        
    print(f"开始推理，共 {len(messages_batch)} 个问题...")
    
    # 限制模型最多输出500个token，temperature设为0保证稳定推理
    sampling_params = SamplingParams(temperature=0.000, max_tokens=500)
    outputs = llm.chat(messages_batch, sampling_params=sampling_params, use_tqdm=True)
    
    # 解析结果
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Relation": case['Rel'],
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\nFinish! {len(df_results)} samples' accuracy: {accuracy:.1f}%\n")
    return df_results


In [70]:
# 运行评测
print("=== Exp: Cross-Modality Dynamic CoT ===")
df_final = run_dynamic_vlm_ablation()
display(df_final)

=== Exp: Cross-Modality Dynamic CoT ===
开始推理，共 50 个问题...


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Finish! 50 samples' accuracy: 94.0%



,Relation,Question,Expected,Model_Output,Correct
0,on,Can a car be on a street?,Yes,"To determine if a car can be on a street, let'...",True
1,wearing,Can a person be wearing a hat?,Yes,"To determine if a person can be wearing a hat,...",True
2,flying in,Can a boat be flying in a room?,No,To determine if a boat can be flying in a room...,True
3,sitting on,Can a cat be sitting on a table?,Yes,To determine if a cat can be sitting on a tabl...,True
4,eating,Can a pizza be eating a man?,No,"To determine if a pizza can be eating a man, l...",True
5,parked on,Can a train be parked on a track?,Yes,To determine if a train can be parked on a tra...,True
6,riding,Can a tree be riding on a bike?,No,To determine if a tree can be riding on a bike...,True
7,holding,Can a dog be holding a branch?,Yes,"To determine if a dog can be holding a branch,...",True
8,growing on,Can a flower be growing on a rock?,Yes,To determine if a flower can be growing on a r...,True
9,of,Can a logo be of a banana?,Yes,"To determine if a logo can be of a banana, let...",True


# test dynamic prompts for specific relationship

In [60]:
def format_question2(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    
    # ======= 新增：为了提高精度，进行“强制物理化”修饰 =======
    if rel == 'riding':
        rel_formatted = 'riding on top of' # 强调空间上的“骑在...上面”

    elif rel == 'wears': 
        rel_formatted = 'wearing'
    elif rel == 'says': 
        rel_formatted = 'saying to'
    elif rel == 'has': 
        rel_formatted = 'have'

    # ======= 拼装逻辑 =======
    if rel == 'and':
        return f"Can {sub_f} {rel} {obj_f} appear together?"
    else:
        # rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        # 进一步在句式上强调“主语”对“宾语”做动作
        return f"Can {sub_f} {rel_formatted} {obj_f}?"

In [63]:
system_prompt_visual = """You are an expert in visual commonsense reasoning. 
You will be provided with examples consisting of an image and a question in the form: 'Can a <Subject> be <Relationship> a <Object>?'.
(Note: In the example images, bounding boxes may be present to help you identify the entities, but you must NOT rely on them for your final logical reasoning).

Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
Determine whether the situation is logically possible in the real world according to commonsense knowledge, and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations)..
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment purely on general physical laws, scales, and typical real-world conditions.
Answer step-by-step and end your response strictly with 'Final Answer: Yes' or 'Final Answer: No'."""


def run_dynamic_vlm_ablation2():
    messages_batch = []
    
    for case in test_cases:
        target_rel = case['Rel']
        question = format_question2(case['Sub'], target_rel, case['Obj'])
        
        # 1. 放入 System Prompt
        messages = [{"role": "system", "content": system_prompt_visual}]
        
        # 2. 动态检索并拼接 针对当前关系的 图像 Few-Shot
        dynamic_shots = get_dynamic_few_shots(target_rel, IMAGE_BASE_DIR, num_shots=FEW_SHOT_COUNT)
        if dynamic_shots:
            messages.extend(dynamic_shots)
            
        # 3. 放入真正的测试题（纯文本！这代表了真实世界的无框提问）
        messages.append({
            "role": "user", 
            "content": [{"type": "text", "text": question}]
        })
        # messages.append({
        #     "role": "user", 
        #     "content": "Can an arm be flying in the sky?"
        # })
        # messages.append(
        #     {"role": "assistant", 
        #      "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"}
        # )
        
        messages_batch.append(messages)
        
    print(f"开始推理，共 {len(messages_batch)} 个问题...")
    
    # 限制模型最多输出500个token，temperature设为0保证稳定推理
    sampling_params = SamplingParams(temperature=0.000, max_tokens=500)
    outputs = llm.chat(messages_batch, sampling_params=sampling_params, use_tqdm=True)
    
    # 解析结果
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Relation": case['Rel'],
            "Question": format_question2(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\nFinish! {len(df_results)} samples' accuracy: {accuracy:.1f}%\n")
    return df_results

In [64]:
# 运行评测
print("=== Exp: Cross-Modality Dynamic CoT ===")
df_final = run_dynamic_vlm_ablation()
display(df_final)

=== Exp: Cross-Modality Dynamic CoT ===
开始推理，共 50 个问题...


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Finish! 50 samples' accuracy: 94.0%



,Relation,Question,Expected,Model_Output,Correct
0,on,Can a car be on a street?,Yes,"To determine if a car can be on a street, let'...",True
1,wearing,Can a person be wearing a hat?,Yes,"To determine if a person can be wearing a hat,...",True
2,flying in,Can a boat be flying in a room?,No,To determine if a boat can be flying in a room...,True
3,sitting on,Can a cat be sitting on a table?,Yes,To determine if a cat can be sitting on a tabl...,True
4,eating,Can a pizza be eating a man?,No,"To determine if a pizza can be eating a man, l...",True
5,parked on,Can a train be parked on a track?,Yes,To determine if a train can be parked on a tra...,True
6,riding,Can a tree be riding a bike?,No,"To determine if a tree can be riding a bike, l...",True
7,holding,Can a dog be holding a branch?,Yes,"To determine if a dog can be holding a branch,...",True
8,growing on,Can a flower be growing on a rock?,Yes,To determine if a flower can be growing on a r...,True
9,of,Can a logo be of a banana?,Yes,"To determine if a logo can be of a banana, let...",True
